# Elo & Bradley–Terry: hands-on (optional)
**LLMs, Animated · Lesson M4.5**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

You'll implement Elo, run it on simulated "Arena" battles between four models, see that its answer depends on the **order** of the battles, then fit **Bradley–Terry** to all battles at once and put **confidence intervals** on the ratings.

Stuck? Solutions are at the very bottom. Only `numpy` and `matplotlib` are needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("ready")

## 1 · Expected score
Elo predicts A's expected score (its win probability, counting a tie as ½) from the rating gap:

$$E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}}$$

In [ ]:
def expected_score(ra, rb):
    # TODO: return E_A for ratings ra (model A) and rb (model B)
    raise NotImplementedError("Exercise 1")

**Check 1**

In [ ]:
assert abs(expected_score(1500, 1500) - 0.5) < 1e-12
assert abs(expected_score(1600, 1500) - 0.6400650) < 1e-6
assert abs(expected_score(1900, 1500) - 10 / 11) < 1e-12, "a 400-point lead should mean 10-to-1 odds"
assert abs(expected_score(1600, 1500) + expected_score(1500, 1600) - 1) < 1e-12
print("✓ exercise 1")

## 2 · The Elo update
After a battle, A's rating moves by the surprise, and B moves by the same amount in the other direction:

$$R_A \leftarrow R_A + K\,(S_A - E_A), \qquad R_B \leftarrow R_B - K\,(S_A - E_A)$$

where $S_A$ is 1 if A won, 0 if A lost, ½ for a tie.

In [ ]:
def elo_update(ra, rb, score_a, k=32):
    # TODO: return the new (ra, rb)
    raise NotImplementedError("Exercise 2")

**Check 2.** These are the lesson's worked example and faded example.

In [ ]:
ra, rb = elo_update(1600, 1500, 0, k=32)          # upset: B wins
assert abs(ra - 1579.518) < 1e-3 and abs(rb - 1520.482) < 1e-3
ra, rb = elo_update(1600, 1500, 1, k=32)          # expected: A wins
assert abs(ra - 1611.518) < 1e-3
ra, rb = elo_update(1500, 1700, 1, k=16)          # faded example
assert abs(ra - 1512.156) < 1e-3 and abs(rb - 1687.844) < 1e-3
print("✓ exercise 2")

## 3 · Simulate an arena and run online Elo
Four models with **true** ratings (unknown to the leaderboard). Each battle picks two different models at random; the winner is drawn from the Bradley–Terry / Elo probability. We ignore ties.

In [ ]:
TRUE = np.array([1000.0, 1100.0, 1150.0, 1300.0])
NAMES = ["model-0", "model-1", "model-2", "model-3"]
M = len(TRUE)

def simulate_battles(n, seed=0):
    rng = np.random.default_rng(seed)
    a = rng.integers(0, M, n)
    b = (a + rng.integers(1, M, n)) % M          # a different model
    p_a = 1 / (1 + 10 ** ((TRUE[b] - TRUE[a]) / 400))
    y = (rng.random(n) < p_a).astype(float)      # 1 if a won
    return a, b, y

a, b, y = simulate_battles(6000)
print("first five battles (a, b, a_won):", list(zip(a[:5].tolist(), b[:5].tolist(), y[:5].astype(int).tolist())))

Implement `run_elo`: start every model at `init`, apply `elo_update` to each battle **in order**, and return the final ratings plus the history (one row per step, including the start).

In [ ]:
def run_elo(a, b, y, n_models, k=32, init=1000.0):
    r = np.full(n_models, init)
    history = [r.copy()]
    # TODO: loop over battles (i, j, s) and update r[i], r[j]; append a copy of r each time
    raise NotImplementedError("Exercise 3")
    return r, np.array(history)

**Check 3.** We also replay the *same* battles in a shuffled order.

In [ ]:
final, hist = run_elo(a, b, y, M, k=32)
assert hist.shape == (len(y) + 1, M)
assert abs(final.sum() - 1000 * M) < 1e-6, "Elo is zero-sum: the total never changes"
assert np.argmax(final) == 3 and np.argmin(final) == 0
# models 1 and 2 are only 50 points apart: with K = 32 their order can flip from one battle to the next

perm = np.random.default_rng(1).permutation(len(y))
final_shuffled, _ = run_elo(a[perm], b[perm], y[perm], M, k=32)
print("Elo, original order:", final.round(1))
print("Elo, shuffled order:", final_shuffled.round(1))
assert np.abs(final - final_shuffled).max() > 1, "same votes, different order → different ratings"

plt.figure(figsize=(7, 3.5))
for m in range(M):
    plt.plot(hist[:, m], lw=1, label=NAMES[m])
    plt.axhline(TRUE[m] - TRUE.mean() + 1000, ls=":", color="gray")
plt.xlabel("battle"); plt.ylabel("Elo rating"); plt.legend(); plt.title("Online Elo, K = 32 (dotted: true, centred)")
plt.show()
print("✓ exercise 3")

## 4 · Fit Bradley–Terry to all battles at once
Bradley–Terry gives each model a strength $\beta_i$ with $P(i \text{ beats } j) = \sigma(\beta_i - \beta_j)$. Fit it by **gradient ascent** on the average log-likelihood. For one battle between $a$ and $b$ with outcome $y$ (1 if $a$ won), the gradient is $(y - p)$ for $\beta_a$ and $-(y - p)$ for $\beta_b$, where $p = \sigma(\beta_a - \beta_b)$.

Then convert to the Elo scale, $R = \beta \cdot 400 / \ln 10$, and anchor model 0 at 1000 (ratings are only defined up to a shift).

*Hint:* `np.add.at(g, a, r)` adds `r[k]` into `g[a[k]]`, even when indices repeat.

In [ ]:
def fit_bt(a, b, y, n_models, steps=2000, lr=2.0):
    beta = np.zeros(n_models)
    # TODO: repeat `steps` times: compute p for every battle, accumulate the gradient g, take a step beta += lr * g / len(y)
    raise NotImplementedError("Exercise 4")
    R = beta * 400 / np.log(10)
    return R - R[0] + 1000

**Check 4.** BT should land close to the truth (relative to model 0), and the battle order should make **no** difference.

In [ ]:
bt = fit_bt(a, b, y, M)
bt_shuffled = fit_bt(a[perm], b[perm], y[perm], M)
print("true (anchored):", TRUE - TRUE[0] + 1000)
print("Bradley–Terry  :", bt.round(1))
assert np.abs(bt - (TRUE - TRUE[0] + 1000)).max() < 30
assert np.allclose(bt, bt_shuffled), "BT uses all votes at once, so order cannot matter"
print("✓ exercise 4")

## 5 · Confidence intervals by bootstrap
Resample the battles **with replacement**, refit BT, repeat, and take the 2.5th and 97.5th percentiles of each model's rating. This is how Arena-style leaderboards get their ± numbers.

In [ ]:
def bootstrap_bt(a, b, y, n_models, n_boot=100, seed=0):
    rng = np.random.default_rng(seed)
    fits = []
    # TODO: n_boot times: draw len(y) random indices with replacement, refit with fit_bt(..., steps=1000), store
    raise NotImplementedError("Exercise 5")
    fits = np.array(fits)
    return np.percentile(fits, 2.5, axis=0), np.percentile(fits, 97.5, axis=0)

**Check 5**

In [ ]:
lo, hi = bootstrap_bt(a, b, y, M)
# model-0 is the anchor (fixed at 1000), so its interval is zero by construction
for m in range(M):
    print(f"{NAMES[m]}: {bt[m]:7.1f}   95% CI [{lo[m]:7.1f}, {hi[m]:7.1f}]")
assert np.all(lo <= bt + 1e-6) and np.all(bt <= hi + 1e-6)
assert np.all(hi[1:] - lo[1:] < 60) and np.all(hi[1:] - lo[1:] > 5)
# with only 300 battles the intervals are much wider
a3, b3, y3 = a[:300], b[:300], y[:300]
lo3, hi3 = bootstrap_bt(a3, b3, y3, M, n_boot=60)
print("CI widths, 6000 battles:", (hi - lo)[1:].round(0), "  300 battles:", (hi3 - lo3)[1:].round(0))
assert np.all((hi3 - lo3)[1:] > (hi - lo)[1:])
print("✓ exercise 5")

## Experiment
Try these and predict the result before you run them:
- Run Elo with `k=4` and `k=64` on the same battles. Which one ends closer to the truth? Which is noisier along the way?
- Make model-1 and model-2 only 10 points apart in `TRUE`. How many battles until their bootstrap intervals stop overlapping?
- Add a "style bias": whenever model-3 is in a battle, flip 5% of its losses into wins. BT faithfully learns the biased preference. Nothing in the maths can tell style from substance.

In [ ]:
# your experiments here

## Recap
- Elo predicts with $E_A = 1/(1+10^{(R_B-R_A)/400})$ and moves ratings by $K(S_A - E_A)$. It depends on the battle order and on $K$.
- Bradley–Terry fits the same model to all votes at once (logistic regression). It doesn't depend on order, and it's what modern leaderboards (and reward models) use.
- Bootstrapping the votes gives confidence intervals. Fewer votes, wider intervals.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def expected_score(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def elo_update(ra, rb, score_a, k=32):
    ea = expected_score(ra, rb)
    delta = k * (score_a - ea)
    return ra + delta, rb - delta

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def run_elo(a, b, y, n_models, k=32, init=1000.0):
    r = np.full(n_models, init)
    history = [r.copy()]
    for i, j, s in zip(a, b, y):
        r[i], r[j] = elo_update(r[i], r[j], s, k)
        history.append(r.copy())
    return r, np.array(history)

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def fit_bt(a, b, y, n_models, steps=2000, lr=2.0):
    beta = np.zeros(n_models)
    for _ in range(steps):
        p = 1 / (1 + np.exp(-(beta[a] - beta[b])))
        r = y - p
        g = np.zeros(n_models)
        np.add.at(g, a, r)
        np.add.at(g, b, -r)
        beta += lr * g / len(y)
    R = beta * 400 / np.log(10)
    return R - R[0] + 1000

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def bootstrap_bt(a, b, y, n_models, n_boot=100, seed=0):
    rng = np.random.default_rng(seed)
    fits = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        fits.append(fit_bt(a[idx], b[idx], y[idx], n_models, steps=1000))
    fits = np.array(fits)
    return np.percentile(fits, 2.5, axis=0), np.percentile(fits, 97.5, axis=0)